In [ ]:
import gymnasium
import numpy as np
from gymnasium.spaces import Discrete, MultiDiscrete, Box, Dict
from gymnasium.utils import seeding

from pettingzoo import AECEnv
from pettingzoo.utils import AgentSelector, wrappers

# Carbon Farming — Action & Observation Spaces

## Farmer Action Space (`gymnasium.spaces.Dict`)

Each farmer chooses **3** things per round:

| Key | Space | Description |
|-----|-------|-------------|
| `crop_choice` | `Discrete(3)` | Which crop to plant — `0` Rice, `1` Maize, `2` Cotton |
| `input_choice` | `Discrete(2)` | Input to apply — `0` Fertilizer, `1` Manure |
| `subscribe` | `Discrete(2)` | Opt in to carbon program — `0` No, `1` Yes |

---

## Aggregator Action Space (`gymnasium.spaces.Dict`)

The aggregator sets **4** contract parameters per round:

| Key | Space | Description |
|-----|-------|-------------|
| `offered_price_per_carbon_credit` | `Box(0, 500, shape=(1,))` | Price ($/tonne CO₂) offered to farmers for each verified credit |
| `monitoring_level` | `Discrete(3)` | Monitoring intensity — `0` low, `1` medium, `2` high |
| `upfront_payment` | `Box(0, 1000, shape=(1,))` | One-time payment ($) made to each subscribed farmer |
| `revenue_sharing` | `Box(0, 100, shape=(1,))` | Percentage (%) of credit revenue shared back with the farmer |

---

## Farmer Observation Space (`gymnasium.spaces.Dict`)

What each farmer sees before acting:

| Key | Space | Description |
|-----|-------|-------------|
| `crop_prices` | `Box(0, ∞, shape=(3,))` | Current market prices ($/kg) for Rice, Maize, Cotton |
| `input_costs` | `Box(0, ∞, shape=(2,))` | Cost ($) of Fertilizer and Manure |
| `offered_price_per_carbon_credit` | `Box(0, ∞, shape=(1,))` | Price per carbon credit offered by the aggregator this round |
| `upfront_payment` | `Box(0, ∞, shape=(1,))` | Upfront payment offered by the aggregator |
| `monitoring_level` | `Discrete(3)` | Monitoring level set by the aggregator — `0` low, `1` medium, `2` high |
| `revenue_sharing` | `Box(0, 100, shape=(1,))` | Revenue-sharing percentage offered by the aggregator |
| `period` | `Discrete(num_periods)` | Current period number |

> Farmers see the aggregator's full offer before deciding — this is the key information asymmetry in the game.

---

## Aggregator Observation Space (`gymnasium.spaces.Dict`)

What the aggregator sees before acting:

| Key | Space | Description |
|-----|-------|-------------|
| `carbon_market_price_per_credit` | `Box(0, ∞, shape=(1,))` | Current open-market price ($/tonne CO₂) for carbon credits |
| `previous_enrollment` | `Box(0, 1, shape=(2,))` | Whether each farmer subscribed last round (`0.0` or `1.0`) |
| `previous_carbon_credits` | `Box(0, ∞, shape=(1,))` | Total verified carbon credits generated last round |
| `period` | `Discrete(num_periods)` | Current period number |

> The aggregator only sees **aggregate outcomes** from the previous round, not the farmers' individual crop/input choices.

---

### Space Types Reference
- **`Discrete(n)`** — a single integer from `{0, 1, ..., n-1}`
- **`Box(low, high, shape)`** — a continuous float array of the given shape, bounded by `low` and `high`
- **`Dict({...})`** — a dictionary combining multiple spaces

> The aggregator acts **first** each round; farmers then observe the offer before deciding.

In [ ]:
def env(render_mode=None):
    """
    The env function often wraps the environment in wrappers by default.
    """
    internal_render_mode = render_mode if render_mode != "ansi" else "human"
    env = raw_env(render_mode=internal_render_mode)
    if render_mode == "ansi":
        env = wrappers.CaptureStdoutWrapper(env)
    env = wrappers.AssertOutOfBoundsWrapper(env)
    env = wrappers.OrderEnforcingWrapper(env)
    return env


class raw_env(AECEnv):
    """
    Carbon Farming AEC Environment.

    Game flow each round:
      1. Aggregator acts first — sets offered price per carbon credit,
         monitoring level, upfront payment, and revenue-sharing percentage.
      2. Each farmer observes the aggregator's offer and decides:
         crop choice, input choice (fertilizer / manure), and whether to
         subscribe to the carbon program.
      3. Once all agents have acted, rewards are computed.

    The game runs for `num_periods` rounds, then terminates.
    """

    metadata = {"render_modes": ["human"], "name": "carbon_farming_v0"}

    # ── crop / input parameters ──────────────────────────────────────────
    CROP_NAMES = ["Rice", "Maize", "Cotton"]
    CROP_YIELDS_KG = [500, 400, 300]                # kg produced per period
    CROP_MARKET_PRICES = [10, 20, 30]                # $/kg

    # base carbon sequestration per crop (tonnes CO₂ / period)
    CROP_CARBON_SEQ = [0.5, 0.8, 0.3]

    INPUT_NAMES = ["Fertilizer", "Manure"]
    INPUT_COSTS = [20, 10]
    # multiplier on carbon sequestration (chemical fertilizer penalises, manure boosts)
    INPUT_CARBON_MULT = [0.8, 1.5]

    MONITORING_LEVELS = ["low", "medium", "high"]
    MONITORING_COSTS = [50, 100, 200]
    # fraction of actual credits that get verified
    MONITORING_ACCURACY = [0.6, 0.8, 0.95]

    def __init__(self, render_mode=None, num_periods=10):
        """
        Args:
            render_mode: "human" or None.
            num_periods: number of rounds in one episode.
        """
        super().__init__()

        self.num_periods = num_periods
        self.carbon_market_price_per_credit = 100.0  # $/tonne on the open market

        # --- agents ----------------------------------------------------------
        self.possible_agents = ["aggregator", "farmer_0", "farmer_1"]
        self.agent_name_mapping = {a: i for i, a in enumerate(self.possible_agents)}

        # --- action spaces ---------------------------------------------------
        farmer_action_space = Dict({
            "crop_choice": Discrete(len(self.CROP_NAMES)),       # 0=Rice,1=Maize,2=Cotton
            "input_choice": Discrete(len(self.INPUT_NAMES)),     # 0=Fertilizer,1=Manure
            "subscribe": Discrete(2),                            # 0=No,1=Yes
        })

        aggregator_action_space = Dict({
            "offered_price_per_carbon_credit": Box(low=0.0, high=500.0,  shape=(1,), dtype=np.float32),
            "monitoring_level":                Discrete(len(self.MONITORING_LEVELS)),
            "upfront_payment":                 Box(low=0.0, high=1000.0, shape=(1,), dtype=np.float32),
            "revenue_sharing":                 Box(low=0.0, high=100.0,  shape=(1,), dtype=np.float32),
        })

        self._action_spaces = {
            "aggregator": aggregator_action_space,
            "farmer_0":   farmer_action_space,
            "farmer_1":   farmer_action_space,
        }

        # --- observation spaces ----------------------------------------------
        farmer_observation_space = Dict({
            "crop_prices":                     Box(low=0.0, high=np.inf, shape=(len(self.CROP_NAMES),), dtype=np.float32),
            "input_costs":                     Box(low=0.0, high=np.inf, shape=(len(self.INPUT_NAMES),), dtype=np.float32),
            "offered_price_per_carbon_credit": Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "upfront_payment":                 Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "monitoring_level":                Discrete(len(self.MONITORING_LEVELS)),
            "revenue_sharing":                 Box(low=0.0, high=100.0,  shape=(1,), dtype=np.float32),
            "period":                          Discrete(self.num_periods),
        })

        aggregator_observation_space = Dict({
            "carbon_market_price_per_credit":  Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "previous_enrollment":             Box(low=0.0, high=1.0,    shape=(2,), dtype=np.float32),  # per farmer
            "previous_carbon_credits":         Box(low=0.0, high=np.inf, shape=(1,), dtype=np.float32),
            "period":                          Discrete(self.num_periods),
        })

        self._observation_spaces = {
            "aggregator": aggregator_observation_space,
            "farmer_0":   farmer_observation_space,
            "farmer_1":   farmer_observation_space,
        }

        self.render_mode = render_mode

    # ── space accessors ──────────────────────────────────────────────────
    def observation_space(self, agent):
        return self._observation_spaces[agent]

    def action_space(self, agent):
        return self._action_spaces[agent]

    # ── render ───────────────────────────────────────────────────────────
    def render(self):
        if self.render_mode is None:
            gymnasium.logger.warn("You are calling render method without specifying any render mode.")
            return

        print(f"\n{'='*60}")
        print(f"  Period {self.current_period}/{self.num_periods}")
        print(f"{'='*60}")

        if self.aggregator_action is not None:
            a = self.aggregator_action
            print(f"  Aggregator offer:")
            print(f"    Price/credit : ${a['offered_price_per_carbon_credit'][0]:.2f}")
            print(f"    Monitoring   : {self.MONITORING_LEVELS[int(a['monitoring_level'])]}")
            print(f"    Upfront pay  : ${a['upfront_payment'][0]:.2f}")
            print(f"    Rev-sharing  : {a['revenue_sharing'][0]:.1f}%")

        for fname in ["farmer_0", "farmer_1"]:
            act = self.farmer_actions.get(fname)
            if act is not None:
                crop = self.CROP_NAMES[int(act["crop_choice"])]
                inp  = self.INPUT_NAMES[int(act["input_choice"])]
                sub  = "Yes" if int(act["subscribe"]) else "No"
                print(f"  {fname}: crop={crop}, input={inp}, subscribe={sub}")

        for agent in self.possible_agents:
            print(f"  Reward {agent}: {self.rewards.get(agent, 0):.2f}")
        print(f"{'='*60}\n")

    # ── observe ──────────────────────────────────────────────────────────
    def observe(self, agent):
        if agent.startswith("farmer"):
            agg = self.aggregator_action or {
                "offered_price_per_carbon_credit": np.array([0.0], dtype=np.float32),
                "upfront_payment":                 np.array([0.0], dtype=np.float32),
                "monitoring_level":                0,
                "revenue_sharing":                 np.array([0.0], dtype=np.float32),
            }
            return {
                "crop_prices":                     np.array(self.CROP_MARKET_PRICES, dtype=np.float32),
                "input_costs":                     np.array(self.INPUT_COSTS, dtype=np.float32),
                "offered_price_per_carbon_credit": np.array(agg["offered_price_per_carbon_credit"], dtype=np.float32).reshape(1),
                "upfront_payment":                 np.array(agg["upfront_payment"], dtype=np.float32).reshape(1),
                "monitoring_level":                int(agg["monitoring_level"]),
                "revenue_sharing":                 np.array(agg["revenue_sharing"], dtype=np.float32).reshape(1),
                "period":                          self.current_period,
            }
        else:  # aggregator
            return {
                "carbon_market_price_per_credit":  np.array([self.carbon_market_price_per_credit], dtype=np.float32),
                "previous_enrollment":             np.array(self.prev_enrollment, dtype=np.float32),
                "previous_carbon_credits":         np.array([self.prev_total_credits], dtype=np.float32),
                "period":                          self.current_period,
            }

    # ── close ────────────────────────────────────────────────────────────
    def close(self):
        pass

    # ── reset ────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        if seed is not None:
            self.np_random, self.np_random_seed = seeding.np_random(seed)

        self.agents = self.possible_agents[:]
        self.rewards = {a: 0 for a in self.agents}
        self._cumulative_rewards = {a: 0 for a in self.agents}
        self.terminations = {a: False for a in self.agents}
        self.truncations = {a: False for a in self.agents}
        self.infos = {a: {} for a in self.agents}

        self.current_period = 0

        # bookkeeping for observations
        self.prev_enrollment = [0.0, 0.0]       # per-farmer subscription last round
        self.prev_total_credits = 0.0

        # per-round action storage
        self.aggregator_action = None
        self.farmer_actions = {}

        # aggregator goes first
        self._agent_selector = AgentSelector(self.agents)
        self.agent_selection = self._agent_selector.next()

    # ── step ─────────────────────────────────────────────────────────────
    def step(self, action):
        if (
            self.terminations[self.agent_selection]
            or self.truncations[self.agent_selection]
        ):
            self._was_dead_step(action)
            return

        agent = self.agent_selection
        self._cumulative_rewards[agent] = 0

        # ── store action ────────────────────────────────────────────────
        if agent == "aggregator":
            self.aggregator_action = action
        else:
            self.farmer_actions[agent] = action

        # ── compute rewards once all agents have acted ──────────────────
        if self._agent_selector.is_last():
            self._compute_rewards()

            self.current_period += 1
            self.truncations = {
                a: self.current_period >= self.num_periods for a in self.agents
            }
        else:
            self._clear_rewards()

        # advance to the next agent
        self.agent_selection = self._agent_selector.next()
        self._accumulate_rewards()

        if self.render_mode == "human":
            self.render()

    # ── reward computation ───────────────────────────────────────────────
    def _compute_rewards(self):
        agg = self.aggregator_action
        offered_price = float(agg["offered_price_per_carbon_credit"][0])
        mon_level     = int(agg["monitoring_level"])
        upfront       = float(agg["upfront_payment"][0])
        rev_share_pct = float(agg["revenue_sharing"][0])

        mon_cost     = self.MONITORING_COSTS[mon_level]
        mon_accuracy = self.MONITORING_ACCURACY[mon_level]

        total_verified_credits = 0.0
        aggregator_revenue = 0.0
        aggregator_cost = mon_cost          # monitoring is a per-period fixed cost
        enrollments = []

        for fname in ["farmer_0", "farmer_1"]:
            act = self.farmer_actions[fname]
            crop_idx  = int(act["crop_choice"])
            input_idx = int(act["input_choice"])
            subscribed = int(act["subscribe"])

            # ── farmer economics ────────────────────────────────────────
            crop_revenue = self.CROP_YIELDS_KG[crop_idx] * self.CROP_MARKET_PRICES[crop_idx]
            input_cost   = self.INPUT_COSTS[input_idx]

            # carbon sequestration (tonnes CO₂)
            raw_credits = self.CROP_CARBON_SEQ[crop_idx] * self.INPUT_CARBON_MULT[input_idx]
            verified_credits = raw_credits * mon_accuracy

            farmer_reward = crop_revenue - input_cost

            if subscribed:
                # farmer receives upfront payment + share of carbon credit revenue
                credit_payment = verified_credits * offered_price
                farmer_reward += upfront + (rev_share_pct / 100.0) * credit_payment

                # aggregator sells verified credits on the open market
                aggregator_revenue += verified_credits * self.carbon_market_price_per_credit
                # aggregator pays the farmer
                aggregator_cost += upfront + credit_payment

                total_verified_credits += verified_credits

            enrollments.append(float(subscribed))
            self.rewards[fname] = farmer_reward

        # aggregator reward = market revenue − total costs
        self.rewards["aggregator"] = aggregator_revenue - aggregator_cost

        # update history for next round's observations
        self.prev_enrollment = enrollments
        self.prev_total_credits = total_verified_credits

        # reset per-round storage
        self.aggregator_action = None
        self.farmer_actions = {}
